
# Real Data Gap Evaluation

実データからの最尤推定と外付けHDD上のデータを利用し、深層学習モデルと推移率行列ベース推定の寿命ギャップを比較します。


In [ ]:

import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath("../"))

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from utils.likelihood import Likelihood_diagonal_exp
from utils.formate_matrix_toMLData import matrix_trimer, formate_dataMatrix
from models.model_0929 import DeepSets_varSets_forDiagnel, collate_fn, varSets_Datasets

import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

plt.rcParams["font.size"] = 18
np.set_printoptions(suppress=True, precision=4)


In [ ]:

DATA_NAME = "shoban"
CSV_PATH = f"../real_data/{DATA_NAME}.csv"
MODEL_PATH = "../model_weights/mixed_distribution/mixed_0929.pth"

TIME_SCALE = 1.0
MLE_INITIAL_GUESS = np.array([-0.5, -1.0, -1.5])
OUTLIER_PERCENTILE = 90
EPS = 1e-9

EXTERNAL_DATASETS = {
    "n200": {"path": "/media/user/TRANSCEND/datas/testdata_n200", "ll_use": True},
    "n400": {"path": "/media/user/TRANSCEND/datas/testdata_n400", "ll_use": True},
    "n600": {"path": "/media/user/TRANSCEND/datas/testdata_n600", "ll_use": True},
    "n800": {"path": "/media/user/TRANSCEND/datas/testdata_n800", "ll_use": True},
    "n1000": {"path": "/media/user/TRANSCEND/datas/testdata_n1000", "ll_use": True},
}

VALID_EXTENSIONS = (".csv", ".txt")
IGNORED_PREFIXES = ("._", ".DS_Store", "Thumbs.db")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FORMATER = formate_dataMatrix()

print(f"Using device: {DEVICE}")
print(f"Target CSV: {Path(CSV_PATH).resolve()}")



## 1. 実データの読み込みと前処理


In [ ]:

def preprocess_real_dataframe(df: pd.DataFrame, data_name: str) -> pd.DataFrame:
    df = df.copy()
    if data_name == "suidou":
        df = df.rename(columns={'建設時健全度（1と仮定）': "pre", '調査時健全度': "post", '経過年数': "time"})
    elif data_name == "shoban":
        df = df.rename(columns={'Be(1-4)': "pre", 'Af(1-4)': "post", 'Ins': "time"})
    elif data_name == "Frank":
        def grading(x):
            if x <= 30:
                return 1
            elif x <= 50:
                return 2
            elif x <= 70:
                return 3
            else:
                return 4
        df["Post_IRI_Class"] = df['Post-State IRI'].apply(grading)
        df["Pre_IRI_Class"] = df['Pre-State IRI'].apply(grading)
        df["time"] = df['Inspection Time of PostState IRI'] - df['Inspection Time of Prestate']
        df = df.rename(columns={'Pre_IRI_Class': "pre", 'Post_IRI_Class': "post"})
    elif data_name == "Tunnel":
        df = df.rename(columns={'事前健全度': "pre", '事後健全度': "post", '検査間隔(年)': "time"})
        df = df[["pre", "post", "time"]]
        df = df.dropna()
        df["pre"] = df["pre"].astype(int)
        df["post"] = df["post"].astype(int)
    elif data_name == "RCBridge":
        df = df.rename(columns={0: "pre", 1: "post", 2: "time"})
        df = df[(df["pre"] < 4) & (df["post"] < 5)]

    required = {"pre", "post", "time"}
    if not required.issubset(df.columns):
        missing = required - set(df.columns)
        raise KeyError(f"Missing columns for preprocessing: {missing}")

    cleaned = df[["pre", "post", "time"]].dropna().copy()
    cleaned["pre"] = cleaned["pre"].astype(int)
    cleaned["post"] = cleaned["post"].astype(int)
    cleaned["time"] = cleaned["time"].astype(float)
    cleaned = cleaned[cleaned["time"] > 0]
    return cleaned.reset_index(drop=True)

def load_real_dataset(csv_path: str, data_name: str) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"{csv_path} not found.")
    if data_name == "RCBridge":
        df = pd.read_csv(csv_path, header=None)
    else:
        df = pd.read_csv(csv_path)
    return preprocess_real_dataframe(df, data_name)


In [ ]:

real_df = load_real_dataset(CSV_PATH, DATA_NAME)
print(f"Loaded {len(real_df)} inspection pairs for {DATA_NAME}.")
display(real_df.head())

plt.figure(figsize=(8, 4))
plt.hist(real_df["time"], bins=30, alpha=0.6, color="#4C72B0", edgecolor="black")
plt.xlabel("Inspection interval (years)")
plt.ylabel("Frequency")
plt.title(f"{DATA_NAME} の点検間隔分布")
plt.tight_layout()
plt.show()



## 2. 実データを用いた最尤推定


In [ ]:

states = real_df[["pre", "post"]].to_numpy(dtype=float)
delta_t = (real_df["time"] / TIME_SCALE).to_numpy(dtype=float)

likelihood_input = np.column_stack((states, delta_t))
likelihood = Likelihood_diagonal_exp(likelihood_input)
estimated_Q = likelihood.optimize(MLE_INITIAL_GUESS)
diag_vec = FORMATER.GetOutputVector_byDiagonal(estimated_Q)
estimated_lifespan_years = (1.0 / (diag_vec + EPS)) * TIME_SCALE

REFERENCE_LIFESPAN_VEC = estimated_lifespan_years.copy()
reference_lifespan = pd.Series(
    estimated_lifespan_years,
    index=["State1→2", "State2→3", "State3→4"],
    name="Estimated lifespan (years)",
)

print("推定された期待寿命 (年):")
display(reference_lifespan.to_frame())



## 3. モデル読み込み


In [ ]:

model = DeepSets_varSets_forDiagnel(device=DEVICE).to(DEVICE)
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()
print("DeepSets model loaded.")



## 4. 外付けHDD上のデータ + 推論ヘルパー


In [ ]:

def safe_inverse(values, eps: float = EPS):
    values = np.asarray(values, dtype=float)
    return 1.0 / (values + eps)

def parse_external_file(file_path: Path, ll_use: bool = True):
    with open(file_path, "rb") as f:
        matrix = np.loadtxt(f, delimiter=",")
    trimer = matrix_trimer(matrix)
    if ll_use:
        ll_trm = trimer.trim_transitionRateMatrix(start=4, end=8)
        ll_vec = FORMATER.GetOutputVector_byDiagonal(ll_trm)
        data = trimer.trim_data(start=8)
    else:
        data = trimer.trim_data(start=4)
        ll_vec = np.zeros(3, dtype=float)
    if data.size == 0:
        raise ValueError("No inspection rows found.")
    state_seq = np.stack([data[:, 0], data[:, 1]], axis=0)
    delta_t = data[:, 2]
    return state_seq, delta_t, ll_vec

def load_external_dataset(label: str, cfg: dict):
    directory = Path(cfg["path"])
    ll_use = cfg.get("ll_use", True)
    if not directory.exists():
        print(f"⚠️ {label}: {directory} not found. Skipping.")
        return None
    states, deltas, targets = [], [], []
    for file_path in sorted(directory.iterdir()):
        if file_path.is_dir():
            continue
        if file_path.name.startswith(IGNORED_PREFIXES):
            continue
        if file_path.suffix.lower() not in VALID_EXTENSIONS:
            continue
        try:
            state_seq, delta_t, ll_vec = parse_external_file(file_path, ll_use=ll_use)
        except Exception as exc:
            print(f"  Skipped {file_path.name}: {exc}")
            continue
        states.append(state_seq)
        deltas.append(delta_t)
        targets.append(ll_vec)
    if not states:
        print(f"⚠️ {label}: no valid samples in {directory}.")
        return None
    print(f"Loaded {len(states)} sequences for {label} from {directory}.")
    return states, deltas, targets

def run_model_inference(states, deltas, targets):
    dataset = varSets_Datasets(states, deltas, targets)
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(),
    )
    pred_expectations = []
    ll_expectations = []
    with torch.no_grad():
        for state_batch, delta_batch, target_batch, lengths in dataloader:
            state_batch = state_batch.to(DEVICE)
            delta_batch = delta_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)
            lengths = lengths.to(DEVICE)

            outputs = model(state_batch, delta_batch, lengths)
            pred_expectations.extend(safe_inverse(outputs.cpu().numpy()))
            ll_expectations.extend(safe_inverse(target_batch.cpu().numpy()))
    return np.array(pred_expectations), np.array(ll_expectations)



## 5. データセットごとの推論


In [ ]:

dataset_results = {}

for label, cfg in EXTERNAL_DATASETS.items():
    loaded = load_external_dataset(label, cfg)
    if loaded is None:
        continue
    states, deltas, targets = loaded
    pred_expect, ll_expect = run_model_inference(states, deltas, targets)
    dataset_results[label] = {
        "pred_expect": pred_expect,
        "ll_expect": ll_expect,
    }

if not dataset_results:
    print("No datasets loaded. Please check EXTERNAL_DATASETS paths.")
else:
    for label, res in dataset_results.items():
        print(f"{label}: {res['pred_expect'].shape[0]} samples")



## 6. ギャップ計測と可視化


In [ ]:

gap_storage = {}
reference_vec = REFERENCE_LIFESPAN_VEC

for label, res in dataset_results.items():
    eval_gap = np.abs(res["pred_expect"] - reference_vec)
    ll_gap = np.abs(res["ll_expect"] - reference_vec)
    if eval_gap.size == 0:
        continue
    norms = np.linalg.norm(ll_gap, axis=1)
    if norms.size == 0:
        continue
    threshold = np.percentile(norms, OUTLIER_PERCENTILE)
    mask = norms < threshold
    if mask.sum() == 0:
        mask = np.ones_like(norms, dtype=bool)
    gap_storage[label] = {
        "gap_eval": eval_gap[mask],
        "gap_ll": ll_gap[mask],
    }

print("Trimmed samples per dataset:")
for label, gaps in gap_storage.items():
    print(f"  {label}: {len(gaps['gap_eval'])}")


In [ ]:

plot_rows = []
dim_rows = []

for label, gaps in gap_storage.items():
    for kind_key, values in gaps.items():
        if values.size == 0:
            continue
        for row in values:
            for dim_idx, value in enumerate(row):
                plot_rows.append({"Dataset": label, "Type": kind_key, "Value": value})
                dim_rows.append({"Dataset": label, "Type": kind_key, "Dim": dim_idx, "Value": value})

if plot_rows:
    gap_df = pd.DataFrame(plot_rows)
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=gap_df, x="Dataset", y="Value", hue="Type")
    plt.ylabel("Gap to MLE lifespan (years)")
    plt.title(f"Gap comparison vs. real-data MLE ({100 - OUTLIER_PERCENTILE}% trimmed)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No gap data available for visualization.")


In [ ]:

if dim_rows:
    dim_df = pd.DataFrame(dim_rows)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for dim_idx in range(3):
        ax_df = dim_df[dim_df["Dim"] == dim_idx]
        if ax_df.empty:
            axes[dim_idx].axis("off")
            continue
        sns.boxplot(data=ax_df, x="Dataset", y="Value", hue="Type", ax=axes[dim_idx])
        axes[dim_idx].set_title(f"Dimension {dim_idx + 1}")
        axes[dim_idx].set_xlabel("Dataset")
        axes[dim_idx].set_ylabel("Gap (years)")
    plt.suptitle("Gap detail per degradation dimension")
    plt.tight_layout()
    plt.show()
else:
    print("No per-dimension gap data to plot.")


In [ ]:

summary_rows = []
for label, gaps in gap_storage.items():
    if gaps["gap_eval"].size == 0:
        continue
    summary_rows.append({
        "Dataset": label,
        "gap_eval_mean": gaps["gap_eval"].mean(),
        "gap_eval_median": np.median(gaps["gap_eval"]),
        "gap_ll_mean": gaps["gap_ll"].mean(),
        "gap_ll_median": np.median(gaps["gap_ll"]),
    })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
else:
    print("Summary is empty. Check dataset loading results.")
